# Day 2: データパイプライン & マスキング

| # | セクション | 学ぶこと | 時間 |
|---|-----------|----------|------|
| 1 | データ取り込み | 外部Stage、COPY INTO | 10分 |
| 2 | 半構造化データ | VARIANT型、FLATTEN | 10分 |
| 3 | Dynamic Table | 宣言的パイプライン、自動リフレッシュ | 10分 |
| 4 | PII分類 | カスタムロール、自動タグ付け | 10分 |
| 5 | マスキングポリシー | カラムレベルセキュリティ | 10分 |

---

> **前提条件**: `setup.sql` が実行済みであること

## 事前準備：セッションコンテキストの設定

In [ ]:
USE DATABASE tb_101;
USE ROLE tb_data_engineer;
USE WAREHOUSE tb_de_wh;

---
## 1. 外部Stageからのデータ取り込み

Snowflakeにおける**ステージ**とは、データファイルの保存場所を指定する名前付きオブジェクトです。

| 概念 | 説明 |
|------|------|
| 外部ステージ | S3/Azure Blob/GCSのパスを指す |
| ファイルフォーマット | CSV/JSON/Parquet等のパース方法を定義 |
| COPY INTO | ステージからテーブルにデータをバルクロード |

### Step 1: メニューデータ用のステージとテーブルを作成

In [ ]:
CREATE OR REPLACE STAGE raw_pos.menu_stage
COMMENT = 'Stage for menu data'
URL = 's3://sfquickstarts/frostbyte_tastybytes/raw_pos/menu/'
FILE_FORMAT = public.csv_ff;

CREATE OR REPLACE TABLE raw_pos.menu_staging
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

### Step 2: COPY INTO でデータをロード

In [ ]:
COPY INTO raw_pos.menu_staging
FROM @raw_pos.menu_stage;

### Step 3: ロード結果の確認

In [ ]:
SELECT * FROM raw_pos.menu_staging LIMIT 10;

---
## 2. 半構造化データとVARIANT型

SnowflakeはVARIANT型でJSON等の半構造化データを直接格納・クエリできます。

| 構文 | 用途 |
|------|------|
| `:` (コロン) | JSONキーへのアクセス |
| `[]` (角括弧) | 配列の要素アクセス |
| `::型` | VARIANTからのキャスト |
| `FLATTEN` | 配列/オブジェクトを行に展開 |

### Step 1: VARIANTカラムの中身を確認

In [ ]:
SELECT menu_item_health_metrics_obj
FROM raw_pos.menu_staging
LIMIT 3;

### Step 2: コロン演算子でデータを抽出

In [ ]:
SELECT
    menu_item_name,
    menu_item_health_metrics_obj:menu_item_id::INTEGER AS menu_item_id,
    menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY AS ingredients
FROM raw_pos.menu_staging
LIMIT 10;

### Step 3: FLATTEN で配列を行に展開

`LATERAL FLATTEN` は配列の各要素を個別の行として展開します。

In [ ]:
SELECT
    i.value::STRING AS ingredient_name,
    m.menu_item_health_metrics_obj:menu_item_id::INTEGER AS menu_item_id
FROM
    raw_pos.menu_staging m,
    LATERAL FLATTEN(INPUT => m.menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY) i
LIMIT 20;

---
## 3. Dynamic Table

Dynamic Tableはデータ変換パイプラインを簡素化する強力なツールです。

| 特徴 | 説明 |
|------|------|
| 宣言的構文 | SELECTクエリでデータを定義 |
| 自動リフレッシュ | ソースデータの変更を自動検出・反映 |
| LAG設定 | データ鮮度の目標を指定（例: 1分以内） |

### Step 1: 材料マスターのDynamic Tableを作成

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE harmonized.ingredient
    LAG = '1 minute'
    WAREHOUSE = 'TB_DE_WH'
AS
SELECT
    ingredient_name,
    menu_ids
FROM (
    SELECT DISTINCT
        i.value::STRING AS ingredient_name,
        ARRAY_AGG(m.menu_item_id) AS menu_ids
    FROM
        raw_pos.menu_staging m,
        LATERAL FLATTEN(INPUT => menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY) i
    GROUP BY i.value::STRING
);

In [ ]:
SELECT * FROM harmonized.ingredient LIMIT 10;

### Step 2: 新メニューを追加して自動リフレッシュを確認

サンドイッチトラック「Better Off Bread」が新メニュー「バインミー」を導入しました。

In [ ]:
INSERT INTO raw_pos.menu_staging
SELECT
    10101, 15, 'Sandwiches', 'Better Off Bread',
    157, 'Banh Mi', 'Main', 'Cold Option',
    9.0, 12.0,
    PARSE_JSON('{
      "menu_item_health_metrics": [{
        "ingredients": ["French Baguette", "Mayonnaise", "Pickled Daikon", "Cucumber", "Pork Belly"],
        "is_dairy_free_flag": "N",
        "is_gluten_free_flag": "N",
        "is_healthy_flag": "Y",
        "is_nut_free_flag": "Y"
      }],
      "menu_item_id": 157
    }');

### Step 3: リフレッシュの確認

> 「Query produced no results」と表示される場合は、Dynamic Tableがまだリフレッシュされていません。最大1分お待ちください。

In [ ]:
SELECT * FROM harmonized.ingredient
WHERE ingredient_name IN ('French Baguette', 'Pickled Daikon');

> **DAG確認**: Snowsight左メニュー → Data → TB_101 → HARMONIZED → Dynamic Tables → INGREDIENT をクリックするとパイプラインDAGを可視化できます。

---
## 4. PII自動分類

ここからはガバナンスのセクションです。`customer_loyalty` テーブルには機密性の高い個人情報が含まれています。

Snowflakeの**自動分類**機能で、PIIを自動検出しタグを付与します。

### Step 1: 顧客データのPII確認

In [ ]:
SELECT TOP 10 *
FROM raw_customer.customer_loyalty;

### Step 2: データスチュワードロールの作成

ガバナンス業務専用のカスタムロールを作成します。

In [ ]:
USE ROLE useradmin;

CREATE OR REPLACE ROLE tb_data_steward
    COMMENT = 'Data governance role';

### Step 3: ロールへの権限付与

In [ ]:
USE ROLE securityadmin;

GRANT OPERATE, USAGE ON WAREHOUSE tb_dev_wh TO ROLE tb_data_steward;
GRANT USAGE ON DATABASE tb_101 TO ROLE tb_data_steward;
GRANT USAGE ON ALL SCHEMAS IN DATABASE tb_101 TO ROLE tb_data_steward;
GRANT SELECT ON ALL TABLES IN SCHEMA raw_customer TO ROLE tb_data_steward;
GRANT ALL ON SCHEMA governance TO ROLE tb_data_steward;
GRANT ALL ON ALL TABLES IN SCHEMA governance TO ROLE tb_data_steward;

SET my_user = CURRENT_USER();
GRANT ROLE tb_data_steward TO USER IDENTIFIER($my_user);

### Step 4: PIIタグの作成と分類プロファイルの設定

In [ ]:
USE ROLE accountadmin;

CREATE OR REPLACE TAG governance.pii;
GRANT APPLY TAG ON ACCOUNT TO ROLE tb_data_steward;
GRANT EXECUTE AUTO CLASSIFICATION ON SCHEMA raw_customer TO ROLE tb_data_steward;
GRANT DATABASE ROLE SNOWFLAKE.CLASSIFICATION_ADMIN TO ROLE tb_data_steward;
GRANT CREATE SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE ON SCHEMA governance TO ROLE tb_data_steward;

In [ ]:
USE ROLE tb_data_steward;
USE WAREHOUSE tb_dev_wh;

CREATE OR REPLACE SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE
  governance.tb_classification_profile(
    {
      'minimum_object_age_for_classification_days': 0,
      'maximum_classification_validity_days': 30,
      'auto_tag': true
    });

### Step 5: タグマップの設定と分類の実行

指定したセマンティックカテゴリのカラムに自動でPIIタグを付与します。

In [ ]:
CALL governance.tb_classification_profile!SET_TAG_MAP(
  {'column_tag_map':[
    {
      'tag_name':'tb_101.governance.pii',
      'tag_value':'pii',
      'semantic_categories':['NAME', 'PHONE_NUMBER', 'POSTAL_CODE', 'DATE_OF_BIRTH', 'CITY', 'EMAIL']
    }]});

In [ ]:
CALL SYSTEM$CLASSIFY('tb_101.raw_customer.customer_loyalty', 'tb_101.governance.tb_classification_profile');

### Step 6: 分類結果の確認

In [ ]:
SELECT
    column_name,
    tag_name,
    tag_value
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('raw_customer.customer_loyalty', 'table'))
WHERE tag_name = 'PII'
ORDER BY column_name;

---
## 5. マスキングポリシー

マスキングポリシーは**カラムレベルセキュリティ**を実現します。

| 仕組み | 説明 |
|--------|------|
| ポリシー定義 | CASE式でロールに応じた表示/マスクを制御 |
| タグ連携 | PIIタグが付いた全カラムに自動適用 |
| 透過的 | アプリケーション変更不要、クエリ時に動的適用 |

### Step 1: STRING型用マスキングポリシーの作成

In [ ]:
CREATE OR REPLACE MASKING POLICY governance.mask_string_pii
  AS (original_value STRING)
  RETURNS STRING ->
    CASE
      WHEN CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
        THEN '****MASKED****'
      ELSE original_value
    END;

### Step 2: DATE型用マスキングポリシーの作成

In [ ]:
CREATE OR REPLACE MASKING POLICY governance.mask_date_pii
  AS (original_value DATE)
  RETURNS DATE ->
    CASE
      WHEN CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
        THEN DATE_TRUNC('year', original_value)
      ELSE original_value
    END;

### Step 3: マスキングポリシーをPIIタグに紐付け

タグに紐付けることで、PIIタグが付いた**全てのカラム**に自動適用されます。

In [ ]:
ALTER TAG governance.pii SET
    MASKING POLICY governance.mask_string_pii,
    MASKING POLICY governance.mask_date_pii;

### Step 4: マスキング効果の確認

データスチュワードロール（非特権）でクエリすると、PIIカラムがマスクされます。

In [ ]:
USE ROLE tb_data_steward;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

### Step 5: 管理者ロールではマスクされないことを確認

In [ ]:
USE ROLE tb_admin;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

---
## まとめ

| 学んだこと | ポイント |
|-----------|--------|
| 外部Stage + COPY INTO | S3からのバルクロード、ファイルフォーマット指定 |
| VARIANT / FLATTEN | JSONを直接格納・展開・クエリ |
| Dynamic Table | 宣言的パイプライン、自動リフレッシュ |
| 自動分類 | SYSTEM$CLASSIFYでPIIを自動検出・タグ付け |
| マスキングポリシー | ロールに応じた動的マスキング、タグ連携で一括適用 |

### ストーリー全体の流れ

```
S3 → Stage → COPY INTO → テーブル → Dynamic Table
                                  ↓
                           CLASSIFY → PIIタグ → マスキングポリシー
                                  ↓
                           ロールに応じた表示制御
```

---
## リセット（環境のクリーンアップ）

ハンズオン終了後に環境を元に戻す場合は、以下を実行してください。

In [ ]:
USE ROLE accountadmin;

-- マスキングポリシー解除
ALTER TAG IF EXISTS governance.pii UNSET
    MASKING POLICY governance.mask_string_pii,
    MASKING POLICY governance.mask_date_pii;
DROP MASKING POLICY IF EXISTS governance.mask_string_pii;
DROP MASKING POLICY IF EXISTS governance.mask_date_pii;

-- タグ解除
ALTER TABLE raw_customer.customer_loyalty
  MODIFY
    COLUMN first_name UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN last_name UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN e_mail UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN phone_number UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN postal_code UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN city UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY,
    COLUMN birthday_date UNSET TAG governance.pii, SNOWFLAKE.CORE.PRIVACY_CATEGORY, SNOWFLAKE.CORE.SEMANTIC_CATEGORY;

-- 分類プロファイル削除
DROP SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE IF EXISTS governance.tb_classification_profile;

-- タグ削除
DROP TAG IF EXISTS governance.pii;

-- ロール削除
DROP ROLE IF EXISTS tb_data_steward;

-- Dynamic Table + menu_staging 削除
DROP DYNAMIC TABLE IF EXISTS harmonized.ingredient;
DROP TABLE IF EXISTS raw_pos.menu_staging;
DROP STAGE IF EXISTS raw_pos.menu_stage;

-- ウェアハウスサスペンド
ALTER WAREHOUSE tb_de_wh SUSPEND;